In [29]:
import osmnx as ox
import networkx as nx
import numpy as np
import folium
import pandas as pd

diadiem = "District 3, Ho Chi Minh City, Vietnam"

graph = ox.graph_from_place(diadiem, network_type="drive")

nodes, edges = ox.graph_to_gdfs(graph)

np.random.seed(10)

n = len(edges)

edges["mat_do_xe"] = np.random.randint(50, 250, n)
edges["toc_do_trung_binh"] = np.random.randint(10, 60, n)
edges["risk"] = edges["mat_do_xe"] / edges["toc_do_trung_binh"]

def mauduong(risk):
    if risk >= 10:
        return "red"
    elif risk >= 5:
        return "orange"
    else:
        return "green"

edges["color"] = edges["risk"].apply(mauduong)

for u, v, key, data in graph.edges(keys=True, data=True):
    if (u, v, key) in edges.index:
        mat_do_xe = edges.loc[(u, v, key), "mat_do_xe"]
        toc_do_trung_binh = edges.loc[(u, v, key), "toc_do_trung_binh"]
        risk = edges.loc[(u, v, key), "risk"]

        data["mat_do_xe"] = mat_do_xe
        data["toc_do_trung_binh"] = toc_do_trung_binh
        data["risk"] = risk
        data["risk_weight"] = data["length"] * (1 + risk / 5)

diem_dau = [10.779500, 106.692000]
diem_cuoi = [10.788200, 106.682900]

nut_dau = ox.distance.nearest_nodes(graph, diem_dau[1], diem_dau[0])
nut_cuoi = ox.distance.nearest_nodes(graph, diem_cuoi[1], diem_cuoi[0])

duong_ngan_nhat = nx.shortest_path(graph, nut_dau, nut_cuoi, weight="length")
duong_de_xuat = nx.shortest_path(graph, nut_dau, nut_cuoi, weight="risk_weight")

def tinh_do_dai_duong(duong, loai_trong_so):
    tong = 0

    for i in range(len(duong) - 1):
        u = duong[i]
        v = duong[i + 1]

        canh = graph.get_edge_data(u, v)

        gia_tri_nho_nhat = None

        for key in canh:
            gia_tri = canh[key][loai_trong_so]

            if gia_tri_nho_nhat == None or gia_tri < gia_tri_nho_nhat:
                gia_tri_nho_nhat = gia_tri

        tong = tong + gia_tri_nho_nhat

    return tong

def lay_toa_do(duong):
    toa_do = []

    for nut in duong:
        vi_do = graph.nodes[nut]["y"]
        kinh_do = graph.nodes[nut]["x"]
        toa_do.append([vi_do, kinh_do])

    return toa_do

center_lat = nodes.geometry.y.mean()
center_lon = nodes.geometry.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

folium.GeoJson(
    edges,
    style_function=lambda feature: {
        "color": feature["properties"]["color"],
        "weight": 3,
        "opacity": 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["mat_do_xe", "toc_do_trung_binh", "risk"],
        aliases=["Mat do xe:", "Toc do TB:", "Muc rui ro:"]
    )
).add_to(m)

folium.PolyLine(
    lay_toa_do(duong_ngan_nhat),
    color="blue",
    weight=7,
    opacity=0.8,
    popup="Tuyen duong ngan nhat"
).add_to(m)

folium.PolyLine(
    lay_toa_do(duong_de_xuat),
    color="purple",
    weight=6,
    opacity=0.9,
    dash_array="10",
    popup="Tuyen duong de xuat tranh tac duong"
).add_to(m)

folium.Marker(diem_dau, popup="Diem xuat phat", tooltip="click").add_to(m)

folium.Marker(diem_cuoi, popup="Diem den", tooltip="click").add_to(m)

m